In [1]:
import pandas as pd
import os

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [2]:
# Read the xlsx file
df_fg = pd.read_excel('All_data/functional_groups.xlsx')

# Filter for adult stage and get unique functional groups
unique_groups = df_fg[df_fg['stage'] == 'adult']['functional_group_raw'].unique()

print(unique_groups)

<ArrowStringArray>
[                                                                    'benthos',
                                                       'benthos > endobenthos',
          'benthos > endobenthos; benthos > epibenthos; benthos > meiobenthos',
                                'benthos > endobenthos; benthos > meiobenthos',
                                                        'benthos > epibenthos',
                                'benthos > epibenthos; benthos > macrobenthos',
                                 'benthos > epibenthos; benthos > meiobenthos',
                                                      'benthos > macrobenthos',
                               'benthos > macrobenthos; benthos > megabenthos',
                               'benthos > macrobenthos; benthos > meiobenthos',
                            'benthos > macrobenthos; plankton > macroplankton',
                                                       'benthos > megabenthos',
                     

Manually go through the functional groups file and label which aphiaIDs should be removed

# Remove the flagged aphiaIDs from the parquet file

In [4]:
# Read the manual removal list
df_manual = pd.read_excel('All_data/functional_groups_manual.xlsx')

# Get AphiaIDs marked for removal
remove_ids = df_manual[df_manual['remove'] == 'x']['aphiaID'].unique()

# Read the parquet file
df_merged = pd.read_parquet('All_data/merged_all.parquet')

# Count before
count_before = len(df_merged)

# Remove records with aphiaID in remove_ids
df_filtered = df_merged[~df_merged['aphiaID'].isin(remove_ids)]

# Count after
count_after = len(df_filtered)

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

# Save filtered parquet
df_filtered.to_parquet('All_data/merged_all_filtered.parquet')

Records before: 3985257
Records after: 3959521
Records removed: 25736


In [5]:
# Use functional_groups_manual.xlsx "remove" flags to filter validated_nomenclature_manual.xlsx by AphiaID
df_validated = pd.read_excel("All_data/validated_nomenclature_manual.xlsx")

# Get AphiaIDs to remove from the manual file (prefer existing df_manual if available)
if "df_manual" in globals():
    df_remove_source = df_manual
else:
    df_remove_source = pd.read_excel("All_data/functional_groups_manual.xlsx")

# Handle possible AphiaID column naming differences
manual_id_col = "AphiaID" if "AphiaID" in df_remove_source.columns else "aphiaID"
validated_id_col = "AphiaID" if "AphiaID" in df_validated.columns else "aphiaID"

ids_to_remove = (
    df_remove_source.loc[df_remove_source["remove"].astype(str).str.strip().str.lower() == "x", manual_id_col]
    .dropna()
    .unique()
)

count_before_validated = len(df_validated)
df_validated_filtered = df_validated[~df_validated[validated_id_col].isin(ids_to_remove)].copy()
count_after_validated = len(df_validated_filtered)

df_validated_filtered.to_excel("All_data/validated_nomenclature_manual_filtered.xlsx", index=False)

print(f"Validated records before: {count_before_validated}")
print(f"Validated records after: {count_after_validated}")
print(f"Validated records removed: {count_before_validated - count_after_validated}")

Validated records before: 13203
Validated records after: 12680
Validated records removed: 523


# Filter out all terrestrial, freshwater and extinct taxa

In [6]:
# Read validated nomenclature file
df_validated_final = pd.read_excel("All_data/validated_nomenclature_final.xlsx")

# Identify the AphiaID column
validated_id_col = "AphiaID" if "AphiaID" in df_validated_final.columns else "aphiaID"

# Get AphiaIDs marked for removal
ids_to_remove = (
    df_validated_final.loc[
        df_validated_final["remove"].astype(str).str.strip().str.lower() == "x",
        validated_id_col
    ]
    .dropna()
    .unique()
)

# Read the filtered parquet file
df_merged_filtered = pd.read_parquet("All_data/merged_all_filtered.parquet")

# Remove matching records
count_before = len(df_merged_filtered)
df_merged_filtered = df_merged_filtered[~df_merged_filtered["aphiaID"].isin(ids_to_remove)].copy()
count_after = len(df_merged_filtered)

# Save back with the same name
df_merged_filtered.to_parquet("All_data/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 3959521
Records after: 3953153
Records removed: 6368


In [ ]:
# Add acceptedAphiaID to the filtered merged parquet using AphiaID -> valid_AphiaID mapping

if "df_validated_final" not in globals():
    df_validated_final = pd.read_excel("All_data/validated_nomenclature_final.xlsx")

validated_map = (
    df_validated_final[["AphiaID", "valid_AphiaID"]]
    .dropna(subset=["AphiaID"])
    .drop_duplicates(subset=["AphiaID"])
    .set_index("AphiaID")["valid_AphiaID"]
)

df_merged_filtered["acceptedAphiaID"] = df_merged_filtered["aphiaID"].map(validated_map)

# Move acceptedAphiaID to be immediately after aphiaID (if both exist)
cols = list(df_merged_filtered.columns)
if "acceptedAphiaID" in cols and "aphiaID" in cols:
    cols.remove("acceptedAphiaID")
    aph_idx = cols.index("aphiaID")
    cols.insert(aph_idx + 1, "acceptedAphiaID")
    df_merged_filtered = df_merged_filtered[cols]

df_merged_filtered.to_parquet("All_data/merged_all_filtered.parquet")

print("acceptedAphiaID column added and saved to All_data/merged_all_filtered.parquet")
print("Missing acceptedAphiaID values:", df_merged_filtered["acceptedAphiaID"].isna().sum())

acceptedAphiaID column added and saved to All_data/merged_all_filtered.parquet
Missing acceptedAphiaID values: 0
